# API-232 Lab (SOLUTIONS): Building a Fantasy Football Team Under a Salary Cap
### Linear programming, binding constraints, the optimizer's curse and matching, with real NFL data

In **fantasy football**, you pick real NFL players for your team and score points based on what they do on the field each week: yards, catches and touchdowns. In this lab you'll act as a fantasy general manager with a twist: you must pay each player his **real NFL salary**, and your total payroll can't exceed a cap.

That turns team-building into the same kind of problem as Quiz 3's vaccination drive: **maximize an objective subject to resource constraints.** Here the objective is projected fantasy points, and the constraints are a salary cap and roster rules.

Then we'll ask two questions that matter as much in policy as in fantasy football:

* **Does the "optimal" plan deliver what it promised?** When you pick the options with the best forecasts, you also pick the ones whose forecasts were most likely too high.
* **How should you assign scarce resources when several people want the same thing?** Fantasy leagues assign newly popular players through a "waiver wire", a small matching market like the summer-jobs question on the quiz.

**Concepts from Quiz 3:**

| Part | Concept | Quiz link |
|---|---|---|
| 1 | Objective, constraints, feasibility, binding constraints | Question 2(a)–(c) |
| 2 | Solving the problem; the value of relaxing a constraint | Question 2(c) |
| 3 | Can adding a rule ever help? | Question 2(d) |
| 4 | The optimizer's curse: forecasts of the chosen plan are too optimistic | New: prediction meets optimization (Question 1) |
| 5 | Boston mechanism vs deferred acceptance, stability, strategy | Question 3 |

**Time:** about 75 minutes. Answer the written questions in the markdown cells marked **✏️ Your answer**.


## 0. Setup (about 3 minutes)

Run the next cell. It installs `nflreadpy`, the Python package for the free [nflverse](https://github.com/nflverse) NFL data, and downloads two tables:

* **Season statistics** (2013–2025) for every quarterback, running back, wide receiver and tight end, including fantasy points.
* **Contracts:** every NFL contract since about 2011, from OverTheCap, including the average salary per year (`apy`, in millions of dollars).

It takes about a minute.

In [ ]:
!pip install -q nflreadpy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy.optimize import milp, linprog, LinearConstraint, Bounds
import nflreadpy as nfl

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

contracts = nfl.load_contracts().to_pandas()
stats = nfl.load_player_stats(list(range(2013, 2026)), summary_level="reg").to_pandas()
stats = stats[stats.position.isin(["QB", "RB", "WR", "TE"])]
print(f"{len(contracts):,} contracts, {len(stats):,} player-seasons")

### The player pool (provided)

`build_pool(season)` returns every player we can pick for a season, with three numbers:

* **`salary`**: his real NFL salary that season, in millions of dollars (the average per year of the contract in force).
* **`proj`**: his **projected** fantasy points, made the simplest way: last season's points per game × 17 games. This is the forecast you have when you build your team before the season.
* **`actual`**: the fantasy points he **actually** scored that season (0 if he didn't play). You don't get to see this until Part 4.

Points use standard "PPR" scoring (a point per catch, a point per 10 yards, 6 per touchdown). Only players who played at least 6 games last season are included.

In [ ]:
def build_pool(season):
    prev = stats[(stats.season == season - 1) & (stats.games >= 6)]
    pool = prev[["player_id", "player_display_name", "position", "recent_team", "games", "fantasy_points_ppr"]].rename(
        columns={"player_display_name": "player", "recent_team": "team_prev",
                 "games": "games_prev", "fantasy_points_ppr": "points_prev"})
    pool["proj"] = (pool.points_prev / pool.games_prev * 17).round(1)

    # the contract in force during `season`
    c = contracts[contracts.gsis_id.notna() & (contracts.apy > 0) & (contracts.year_signed <= season)
                  & (contracts.year_signed + contracts.years.fillna(1) > season)]
    c = c.sort_values("year_signed").groupby("gsis_id").tail(1)
    pool = pool.merge(c[["gsis_id", "apy"]], left_on="player_id", right_on="gsis_id").rename(columns={"apy": "salary"})
    pool["salary"] = pool.salary.round(2)

    this = stats[stats.season == season].set_index("player_id")
    pool["team"] = pool.player_id.map(this.recent_team).fillna(pool.team_prev)
    pool["actual"] = pool.player_id.map(this.fantasy_points_ppr).fillna(0).round(1)
    pool["season"] = season
    return (pool[["season", "player", "position", "team", "salary", "games_prev", "proj", "actual"]]
            .sort_values("proj", ascending=False).reset_index(drop=True))

pool = build_pool(2025)
print(len(pool), "players")
pool.head(10)

## Part 1. Writing down the problem (about 12 minutes)

**The rules.** Your team has exactly **8 players**:

| Slot | Requirement |
|---|---|
| Quarterback (QB) | exactly 1 |
| Running back (RB) | 2 or 3 |
| Wide receiver (WR) | 3 or 4 |
| Tight end (TE) | 1 or 2 |
| Total | exactly 8 (the 8th player is a "flex": an extra RB, WR or TE) |

Your total salary can't exceed **$50 million**. Your goal is the most projected fantasy points.

**1a.** Write the objective and the constraints.

✅ **Answer:**

Maximize $\sum_i p_i x_i$ subject to:

* Salary cap: $\sum_i s_i x_i \le 50$
* $\sum_{i \in QB} x_i = 1$
* $2 \le \sum_{i \in RB} x_i \le 3$
* $3 \le \sum_{i \in WR} x_i \le 4$
* $1 \le \sum_{i \in TE} x_i \le 2$
* $\sum_i x_i = 8$
* $x_i \in \{0, 1\}$

This is the vaccination problem with more variables: an objective that adds up value, and constraints that add up resource use. One difference matters later: on the quiz, fractional station-hours were fine. Here you can't sign two-thirds of a player, so $x_i$ must be 0 or 1. That makes this an **integer** program.

**1b.** Four friends propose teams. For each one, compute the total salary, projected points and the number of players at each position. Then decide whether it is feasible.

In [ ]:
proposals = {
    "A: last year's stars": ["Lamar Jackson", "Saquon Barkley", "Jahmyr Gibbs", "Bijan Robinson",
                             "Ja'Marr Chase", "Chris Godwin Jr.", "Puka Nacua", "George Kittle"],
    "B: rookie contracts":  ["Bo Nix", "Jahmyr Gibbs", "De'Von Achane", "Kenneth Walker III",
                             "Puka Nacua", "Brian Thomas Jr.", "Ladd McConkey", "Brock Bowers"],
    "C: two quarterbacks":  ["Jayden Daniels", "Bo Nix", "Jahmyr Gibbs", "Bijan Robinson",
                             "Puka Nacua", "Malik Nabers", "Brian Thomas Jr.", "Brock Bowers"],
    "D: best points per $": ["Aidan O'Connell", "Ameer Abdullah", "Audric Estimé", "Puka Nacua",
                             "Ray-Ray McCloud", "Kevin Austin Jr.", "Jordan Akins", "Josh Whyle"],
}

def summarize(names, pool=pool):
    team = pool[pool.player.isin(names)]
    out = {"salary": round(team.salary.sum(), 2), "proj": round(team.proj.sum(), 1)}
    for pos in ["QB", "RB", "WR", "TE"]:
        out[pos] = int((team.position == pos).sum())
    return out

pd.DataFrame({name: summarize(names) for name, names in proposals.items()}).T

**1c.** Which proposals are feasible, which is best, and which constraints bind?

✅ **Answer:**

* **A is infeasible.** It costs about $165 million, more than three times the cap. It projects the most points, but that doesn't matter if you can't afford it.
* **B is feasible:** $24.3 million, 1 QB, 3 RBs, 3 WRs, 1 TE. It projects 2,382 points.
* **C is infeasible**, even though it costs only $40.6 million. It has two quarterbacks, and the flex slot must be a RB, WR or TE.
* **D is feasible** but projects only 1,067 points.

So **B** is the best feasible proposal. Its binding constraints are the QB rule (exactly 1), the total-players rule (exactly 8), the RB maximum (3 of 3), the WR minimum (3 of 3) and the TE minimum (1 of 1). The **salary cap is not binding**: B leaves about $26 million unspent. That's a hint B can be improved, since unused budget could buy a better player.

**1d.** Why doesn't "best points per dollar" give the best team?

✅ **Answer:** Points per dollar is the right idea when you can buy any amount of anything, but here you have only 8 roster spots. D fills them with very cheap, low-scoring players and leaves almost all the money unspent ($3.8 million of $50 million). Once roster spots are the scarce resource, a player who uses a spot but scores little is expensive, however cheap his salary. Greedy rules that rank by a ratio ignore that some constraints are about **count**, not money.

## Part 2. Solving it (about 15 minutes)

Checking proposals by hand doesn't scale: there are hundreds of millions of possible teams. A **solver** searches them efficiently. `scipy.optimize.milp` solves problems of the form:

> minimize $c \cdot x$ subject to $\text{lower} \le A x \le \text{upper}$, with chosen $x$ restricted to integers

Each **row** of $A$ is one constraint: it gives, for each player, how much of that resource he uses. `milp` *minimizes*, so to maximize points we minimize **negative** points.

**2a.** Fill in the constraint rows below. The salary row is done for you.

In [ ]:
SLOTS = {"QB": (1, 1), "RB": (2, 3), "WR": (3, 4), "TE": (1, 2)}   # (min, max) of each position

def solve_lineup(pool, cap, score="proj", extra_rules=(), relax=False):
    """Returns x: a 0/1 (or fractional, if relax=True) choice for each player.
    extra_rules: a list of (row, lower, upper) tuples added as constraints (used in Part 3)."""
    n = len(pool)
    rows, lo, hi = [pool.salary.values], [0], [cap]          # salary: 0 <= total salary <= cap

    for pos, (a, b) in SLOTS.items():
        rows.append((pool.position == pos).values.astype(float)); lo.append(a); hi.append(b)
    rows.append(np.ones(n)); lo.append(8); hi.append(8)

    assert len(rows) == 6, "Add the four position rows and the roster-size row before solving"
    for r, a, b in extra_rules:
        rows.append(np.asarray(r, float)); lo.append(a); hi.append(b)
    res = milp(c=-pool[score].values, constraints=LinearConstraint(np.vstack(rows), lo, hi),
               integrality=np.zeros(n) if relax else np.ones(n), bounds=Bounds(0, 1))
    return res.x

CAP = 50
x = solve_lineup(pool, CAP)
best = pool[x > 0.5]
print(f"Salary ${best.salary.sum():.2f}M, projected {best.proj.sum():.1f} points")
best

**2b.** Compare with proposal B. What kind of contracts dominate?

✅ **Answer:** The optimal team projects **2,558.7 points**, 177 more than B, for **$48.15 million**. It swaps in Jayden Daniels at QB, Bijan Robinson and Alvin Kamara at RB and Malik Nabers at WR. Seven of the eight players are on **rookie contracts**. NFL rookie pay is set by a fixed scale based on draft position, not by how well the player turns out. A star in his first four years is therefore paid far below what he'd earn on the open market. The optimizer finds exactly those mispriced players: the same logic as a policy analyst looking for programs whose cost is set by rule rather than by results.

**2c. How much is more cap space worth?** Solve the problem for caps from $5 million to $150 million and plot the best projected points against the cap.

In [ ]:
caps = [5, 10, 15, 20, 25, 30, 40, 50, 60, 80, 100, 125, 150]
value = [pool[solve_lineup(pool, c) > 0.5].proj.sum() for c in caps]

plt.plot(caps, value, marker="o")
plt.xlabel("Salary cap ($ millions)"); plt.ylabel("Best projected points")
plt.title("Each extra dollar of cap buys fewer points"); plt.grid(alpha=0.3); plt.show()
print(pd.Series(value, index=caps).round(1))

**2d. Fractions of players.** On the quiz, fractional station-hours were allowed. What happens if we allow fractional players? Run the **relaxed** problem, where each $x_i$ can be anything between 0 and 1.

In [ ]:
xr = solve_lineup(pool, CAP, relax=True)
relaxed = pool.assign(x=xr.round(3))[xr > 1e-6]
print(f"Relaxed optimum: {(xr * pool.proj).sum():.1f} points, salary ${(xr * pool.salary).sum():.2f}M")
relaxed[["player", "position", "salary", "proj", "x"]]

In a relaxed (fractional) problem, the solver also reports a **shadow price** for each constraint: how much the objective would rise if that constraint were loosened by one unit. The cell below gets the shadow price of the salary cap.

In [ ]:
n = len(pool)
P = lambda pos: (pool.position == pos).values.astype(float)
A_ub = np.vstack([pool.salary.values, P("RB"), P("WR"), P("TE"), -P("RB"), -P("WR"), -P("TE")])
b_ub = [CAP, 3, 4, 2, -2, -3, -1]
A_eq = np.vstack([P("QB"), np.ones(n)])
lp = linprog(-pool.proj.values, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=[1, 8], bounds=(0, 1), method="highs")
shadow = -lp.ineqlin.marginals[0]
print(f"Shadow price of the cap: {shadow:.2f} projected points per extra $1 million")

x51 = solve_lineup(pool, 51)
print(f"Integer optimum at $50M: {best.proj.sum():.1f}   at $51M: {pool[x51 > 0.5].proj.sum():.1f}")

**2e.** Fractions, slack and shadow prices.

✅ **Answer:**

1. The relaxed solution takes about **two-thirds of Saquon Barkley and one-third of De'Von Achane** as its third running back, for 2,587.8 points. You can't sign two-thirds of a player. Rounding either way breaks something: rounding Barkley up and Achane down busts the cap, and the reverse throws away points. Rounded fractional solutions are generally not the integer optimum, which is why the solver has to treat integrality as a constraint.
2. In the **fractional** problem the cap is binding: the solver spends exactly $50 million. In the **integer** problem the best team spends $48.15 million, so the cap has slack, yet raising it to $51 million raises the optimum from 2,558.7 to **2,572.9** (+14.2). The extra money lets you afford a *different combination* of players, even though no single dollar is wasted now. In integer problems, "slack" doesn't mean the constraint doesn't matter. The quiz's rule ("a constraint with slack can be loosened without changing the answer") only holds for continuous problems.
3. The shadow price is about **4.0 points per $1 million**, the average rate of trade-off at the margin. The actual gain from the 51st million is 14.2 points, while the whole step from $40 million to $50 million bought only 22 points (see 2c). In integer problems, the value of more budget comes in lumps. The shadow price is a useful average, not a promise.

## Part 3. Adding rules (about 10 minutes)

On the quiz, the county added a rule: at least 50 station-hours must be mobile vans. Here, a league commissioner is considering two rules.

* **Spread the talent:** no more than one player from any NFL team.
* **Veteran minimum:** at least two players earning $10 million or more, so fantasy teams look like "real" NFL rosters.

**3a.** Can either rule increase the best projected points? Can either leave it unchanged?

✅ **Answer:** Neither can increase it. A new rule only removes teams from the feasible set, and the best team in a smaller set can't beat the best in the bigger one. A rule leaves the optimum unchanged if the current best team already satisfies it. In that case the rule isn't binding and costs nothing.

**3b.** Add each rule with `extra_rules` and compare.

In [ ]:
base = best.proj.sum()

# Rule 1: at most one player per NFL team (one row per team)
spread = [((pool.team == t).values.astype(float), 0, 1) for t in pool.team.unique()]
x1 = solve_lineup(pool, CAP, extra_rules=spread)
print(f"Spread rule:  {pool[x1 > 0.5].proj.sum():.1f} points (cost {base - pool[x1 > 0.5].proj.sum():.1f})")
print("   teams in the original best lineup:", best.team.value_counts().to_dict())

# Rule 2: at least two players with salary >= 10
veteran = [((pool.salary >= 10).values.astype(float), 2, 8)]
x2 = solve_lineup(pool, CAP, extra_rules=veteran)
team2 = pool[x2 > 0.5]
print(f"Veteran rule: {team2.proj.sum():.1f} points (cost {base - team2.proj.sum():.1f})")
team2

**3c.** Which rule binds, what does it cost, and how would you present it?

✅ **Answer:** The **spread rule isn't binding**: the best team already has 8 players from 8 different NFL teams, so the rule costs nothing. The **veteran rule is binding**. The best team now swaps Bijan Robinson for Josh Jacobs and Jayden Daniels for Bo Nix, and projects **2,471.5 points, a cost of about 87 points**. For comparison, the curve in 2c shows that cutting the cap from $50 million to $30 million costs about 115 points, so the rule is almost as costly as losing $20 million of cap.

Present it as a price, like the quiz's mobile-van rule: "This rule has a goal we may value, and it costs about 87 projected points, or about 3% of the team's projected total." Whether that's worth it is a value judgment for the league; the analyst's job is to put a number on the cost and to show that a rule which isn't binding (like the spread rule) is free.

## Part 4. Did the plan deliver? The optimizer's curse (about 18 minutes)

So far we've trusted the projections. Now look at what the players **actually** scored in 2025.

**4a.** Compare your optimal team's projected and actual points.

In [ ]:
print(best[["player", "position", "salary", "proj", "actual"]].to_string(index=False))
print(f"\nProjected {best.proj.sum():.0f}, actual {best.actual.sum():.0f}, "
      f"shortfall {best.proj.sum() - best.actual.sum():.0f} points ({1 - best.actual.sum() / best.proj.sum():.0%})")

**Is that just bad luck?** The optimizer picked the players whose projections were *highest relative to their salary*. But a projection is the truth plus an error. Players whose projections are unusually high are, on average, players whose errors happened to be positive. So the team you pick tends to be the one whose forecast was most over-optimistic, even if each individual forecast is unbiased. This is called the **optimizer's curse**.

**4b. A simulation.** Pretend there are 400 players. Each has a true value, and our projection = true value + random noise with **mean zero**, so every individual projection is unbiased.

In [ ]:
rng = np.random.default_rng(232)
over_best, over_random = [], []
for _ in range(2000):
    truth = rng.normal(200, 50, 400)
    proj = truth + rng.normal(0, 60, 400)          # unbiased: errors average zero
    top = np.argsort(proj)[-8:]
    over_best.append(proj[top].sum() - truth[top].sum())
    rnd = rng.choice(400, 8, replace=False)
    over_random.append(proj[rnd].sum() - truth[rnd].sum())

print(f"Average overestimate, top-8 by projection: {np.mean(over_best):.0f} points")
print(f"Average overestimate, random 8:            {np.mean(over_random):.0f} points")

**4c. Calibrating the forecast.** A fix is to ask: historically, when a player was projected for $p$ points, how many did he actually score? Fit that relationship on **earlier seasons only**, then use it to forecast. The cell below does this for every season from 2016 to 2025. For each season it reports the optimal team's naive projection, its **calibrated** forecast, its actual points, and whether optimizing on the calibrated forecast would have picked a different team.

In [ ]:
pools = {s: build_pool(s) for s in range(2014, 2026)}
rows = []
for s in range(2016, 2026):
    train = pd.concat([pools[t] for t in range(2014, s)])
    calib = smf.ols("actual ~ proj * C(position)", train).fit()   # actual points as a function of projection
    p = pools[s].copy()
    p["calibrated"] = calib.predict(p)
    naive = p[solve_lineup(p, CAP) > 0.5]
    recal = p[solve_lineup(p, CAP, score="calibrated") > 0.5]
    rows.append({"season": s, "projected": naive.proj.sum(), "calibrated": naive.calibrated.sum(),
                 "actual": naive.actual.sum(), "same team?": set(naive.player) == set(recal.player),
                 "actual, calibrated team": recal.actual.sum()})
curse = pd.DataFrame(rows).set_index("season").round(0)
print(curse)
print("\nAverages:"); print(curse.drop(columns="same team?").mean().round(0))

**4d.** Interpretation.

✅ **Answer:**

1. Each projection is right on average, but the top 8 aren't a random 8. They are chosen *because* their projections are high, and a projection is high partly because the true value is high and partly because the error happened to be positive. Selecting on the projection selects on the error, so the chosen group's errors average well above zero (about 880 points in the simulation). The random group's errors average zero.
2. Across 2016–2025, the optimal team was projected for about **2,730 points but scored about 1,870**, roughly 30% less. The calibrated forecast averaged about **1,770**, much closer and slightly conservative. Part of the naive shortfall is the curse, and part is that "last year's points per game × 17" ignores injuries and missed games. Calibration corrects both.
3. Mostly no: the calibrated team was identical to the naive one in 8 of 10 seasons, and in the other two it scored only slightly more (1,877 vs 1,870 points on average across all ten). Calibration pulls high projections down and low ones up, but within a position it roughly preserves the *order*, and the optimizer cares about order and trade-offs, not levels. **Calibration fixes the promise, not necessarily the plan.** To change the plan you need better information (say, about injury risk or who will lose playing time), not just a shrunk version of the same forecast.

**4e.** Why do scaled-up programs disappoint?

✅ **Answer:** This is the optimizer's curse. Each pilot estimate is the true effect plus noise. Picking the three with the largest estimates also picks the ones with the luckiest noise, so their true effects are, on average, smaller than estimated, even if every evaluation was unbiased. The foundation should expect it and plan for it: **shrink** the pilot estimates toward the average effect of similar programs before projecting impact (more for small or noisy pilots), **replicate** the promising pilots before full scale-up, and judge success against the shrunk forecast rather than the pilot's headline number.

## Part 5. The waiver wire as a matching market (about 12 minutes)

During the season, a player who wasn't on anyone's fantasy team can suddenly break out. When several managers want him, the league uses the **waiver wire**: each manager submits a ranked list of claims, and the league has a **priority order**. Usually the worst-placed team has the highest priority, and the same order applies to every player.

After week 5 of 2025, three unclaimed players had just had big games:

In [ ]:
wk = nfl.load_player_stats([2025], summary_level="week").to_pandas()
wk = wk[wk.season_type == "REG"]
names = ["Rico Dowdle", "Kendrick Bourne", "AJ Barner"]
show = wk[wk.player_display_name.isin(names)]
summary = show.groupby(["player_display_name", "position"]).apply(lambda d: pd.Series({
    "avg weeks 1-4": d[d.week < 5].fantasy_points_ppr.mean(),
    "week 5": d[d.week == 5].fantasy_points_ppr.sum()}), include_groups=False).round(1)
summary

Three managers want them. Their rankings (from best to worst) and the waiver priority:

| Manager | 1st choice | 2nd choice | 3rd choice |
|---|---|---|---|
| Ana (priority 1) | Dowdle | Barner | Bourne |
| Ben (priority 2) | Dowdle | Bourne | Barner |
| Cal (priority 3) | Bourne | Dowdle | Barner |

Every player has the same priority over managers: **Ana > Ben > Cal**. The functions below implement both mechanisms from Quiz 3.

In [ ]:
prefs = {"Ana": ["Dowdle", "Barner", "Bourne"],
         "Ben": ["Dowdle", "Bourne", "Barner"],
         "Cal": ["Bourne", "Dowdle", "Barner"]}
priority = ["Ana", "Ben", "Cal"]                     # same order for every player
players = ["Dowdle", "Bourne", "Barner"]

def boston(prefs, priority):
    """Round k: each unassigned manager claims his k-th choice; each player goes to the
    highest-priority claimant if still available. Assignments are final."""
    match, taken = {}, set()
    for k in range(len(players)):
        claims = {}
        for m, ranking in prefs.items():
            if m not in match and ranking[k] not in taken:
                claims.setdefault(ranking[k], []).append(m)
        for p, ms in claims.items():
            winner = min(ms, key=priority.index)
            match[winner] = p; taken.add(p)
    return match

def deferred_acceptance(prefs, priority):
    """Managers apply in order of preference; each player tentatively holds the
    highest-priority applicant so far and rejects the rest, until no one is rejected."""
    nxt = {m: 0 for m in prefs}; held = {}
    free = list(prefs)
    while free:
        m = free.pop(0); p = prefs[m][nxt[m]]; nxt[m] += 1
        if p not in held:
            held[p] = m
        elif priority.index(m) < priority.index(held[p]):
            free.append(held[p]); held[p] = m
        else:
            free.append(m)
    return {m: p for p, m in held.items()}

def justified_envy(match, prefs, priority):
    """Pairs (manager, player) where the manager prefers the player to his own
    and has higher priority than the manager who got him."""
    owner = {p: m for m, p in match.items()}
    out = []
    for m, ranking in prefs.items():
        for p in ranking[:ranking.index(match[m])]:
            if priority.index(m) < priority.index(owner[p]):
                out.append((m, p))
    return out

**5a.** Run both mechanisms. Report each assignment and any justified envy.

In [ ]:
for name, mech in [("Boston", boston), ("Deferred acceptance", deferred_acceptance)]:
    m = mech(prefs, priority)
    print(f"{name:20s} {m}   justified envy: {justified_envy(m, prefs, priority)}")

**5b.** Under the Boston mechanism, can Ben get a player he truly prefers by submitting a different ranking? Try it in code.

In [ ]:
lie = dict(prefs, Ben=["Bourne", "Dowdle", "Barner"])
print("Boston, Ben lists Bourne first:", boston(lie, priority))
print("DA,     Ben lists Bourne first:", deferred_acceptance(lie, priority))

**5c.** Interpretation.

✅ **Answer:**

1. **Boston** gives Ana–Dowdle, Cal–Bourne, Ben–Barner. Ben has **justified envy** of Cal: Ben prefers Bourne to Barner and has higher priority than Cal. **Deferred acceptance** gives Ana–Dowdle, Ben–Bourne, Cal–Barner, with no justified envy, so it is stable.
2. Under Boston, Ben's first-round claim on Dowdle is wasted, since Ana outranks him. By the time he claims Bourne in round 2, Cal has already taken Bourne in round 1. If Ben lists Bourne first, he wins Bourne in round 1 because he outranks Cal. Boston punishes truthful ranking of a popular player you won't get. Under deferred acceptance, lying doesn't help: his tentative claim on Dowdle is rejected, he moves to Bourne, and he still beats Cal. DA is strategy-proof for the managers.
3. With one shared priority order, the top-priority manager is never rejected from his first choice. The second manager is rejected only by players the first holds, and so on. So DA ends with each manager, in priority order, getting his favourite among the players still left: exactly a serial dictatorship.
4. No. Boston "gives" Ana and Cal their first choices only because of the rankings people submitted. Once managers realize that listing a contested player first is risky, they'll stop reporting their true first choices, and the first-choice count stops meaning anything. It also leaves Ben with justified envy. (Same point as the quiz's last question.)

**5d. What happened next?** Run the cell below to see how each player scored for the rest of the season.

In [ ]:
after = show[show.week > 5].groupby("player_display_name").fantasy_points_ppr.mean().round(1)
pd.DataFrame({"week 5": show[show.week == 5].set_index("player_display_name").fantasy_points_ppr,
              "avg after week 5": after})

**5d.** Connect to Part 4.

✅ **Answer:** All three regressed sharply. Dowdle averaged about 13.6 points after his 32-point week, Barner about 8 and Bourne about 5. Bourne, the player Ben fought for, scored the least. A breakout week is a "projection" based on one observation, and the waiver wire picks the most extreme of them, so it's the optimizer's curse again. A good matching mechanism allocates players fairly according to what managers *say* they want. It can't protect managers from wanting the wrong thing, so they should shrink their estimate of a one-week breakout heavily before spending a high waiver priority on him.

## Part 6. Bringing it together (about 5 minutes)

**6.** Advise the housing agency.

✅ **Answer (example):** The outer-borough rule can't increase the number of units the model delivers; at best it costs nothing, if the current plan already meets it. Re-solve with and without the rule and report the difference as the rule's price, for example "2% fewer units for a more even geographic spread", so the council can decide whether the goal is worth it. Because projects can't be funded fractionally, check whether a slightly larger or smaller budget changes the plan in a lumpy way before relying on shadow prices. Finally, expect the funded portfolio to deliver fewer units than estimated, since the model selects the projects whose estimates are most optimistic. Calibrate project estimates against past projects' actual results, and report that calibrated total rather than the model's raw number.